Exploring the different results from bank churn:

In [2]:
import os
import pandas as pd

CSV = os.path.join("data", "churn.csv")


def load():
    df = pd.read_csv(CSV)
    # The raw file has spaces in some headers ("Num Of Products"); normalise.
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    return df


def rate_by(df, col):
    """Churn rate within each group of `col`, sorted worst-first."""
    out = (df.groupby(col)["Churn"]
             .agg(customers="count", churn_rate="mean")
             .sort_values("churn_rate", ascending=False))
    out["churn_rate"] = (out["churn_rate"] * 100).round(1)
    return out


def main():
    df = load()

    print("=" * 60)
    print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
    print(f"Overall churn rate: {df['Churn'].mean() * 100:.1f}%")
    print(f"Missing values: {int(df.isna().sum().sum())}")
    print("=" * 60)

    print("\nChurn rate by GEOGRAPHY (fairness slice #1):")
    print(rate_by(df, "Geography").to_string())

    print("\nChurn rate by GENDER (fairness slice #2):")
    print(rate_by(df, "Gender").to_string())

    print("\nChurn rate by ACTIVE MEMBER (0 = inactive):")
    print(rate_by(df, "IsActiveMember").to_string())

    print("\nChurn rate by NUMBER OF PRODUCTS:")
    print(rate_by(df, "NumOfProducts").to_string())

    print("\nAge — churners vs stayers (mean):")
    print(df.groupby("Churn")["Age"].mean().round(1).to_string())

    print("\nBalance — churners vs stayers (mean):")
    print(df.groupby("Churn")["Balance"].mean().round(0).to_string())

    print("\n" + "=" * 60)
    print("What to notice (your interview talking points):")
    print("- ~1 in 5 customers churn; the classes are imbalanced, so you")
    print("  evaluate with AUC/PR-AUC, not accuracy.")
    print("- Germany churns far more than France/Spain — this is the base-rate")
    print("  gap that later shows up in the fairness audit.")
    print("- Inactive members and customers with 3-4 products churn much more.")
    print("- Churners skew older with higher balances — real, explainable signal.")
    print("=" * 60)

In [5]:
load()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCreditCard,IsActiveMember,EstimatedSalary,Churn
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [3]:

main()

Rows: 10,000   Columns: 13
Overall churn rate: 20.4%
Missing values: 0

Churn rate by GEOGRAPHY (fairness slice #1):
           customers  churn_rate
Geography                       
Germany         2509        32.4
Spain           2477        16.7
France          5014        16.2

Churn rate by GENDER (fairness slice #2):
        customers  churn_rate
Gender                       
Female       4543        25.1
Male         5457        16.5

Churn rate by ACTIVE MEMBER (0 = inactive):
                customers  churn_rate
IsActiveMember                       
0                    4849        26.9
1                    5151        14.3

Churn rate by NUMBER OF PRODUCTS:
               customers  churn_rate
NumOfProducts                       
4                     60       100.0
3                    266        82.7
1                   5084        27.7
2                   4590         7.6

Age — churners vs stayers (mean):
Churn
0    37.4
1    44.8

Balance — churners vs stayers (mean):
C